# EEG activity across brain states

**Task 2: EEG visualization and power spectra — initial checkpoint**

This notebook visualizes surface EEG recordings from mice in different experimental states. For each condition, the intended deliverables are a 30–60 second raw EEG plot using 3–5 channels and a power spectral density (PSD) plot. A spectrogram and stimulus-aligned analysis are included to show how frequency content and evoked responses change over time.

**Current status:** Awake and isoflurane data from subject 688277 have been extracted and verified. Urethane, psilocybin, and ketanserin will be added after the second NWB recording is inspected. This checkpoint documents the completed method, preliminary observations, limitations, and the exact remaining work.

## 1. What EEG measures

EEG records voltage differences at electrodes placed on the skull surface. It mainly reflects electrical fields produced when large populations of nearby neurons receive and process input together. It does not directly record the firing of one neuron.

A raw EEG trace describes **voltage over time**. Frequency analyses are mathematical descriptions calculated from that same voltage recording:

- Raw trace: *What did the voltage do at each moment?*
- Power spectrum: *Which frequencies were strongest on average?*
- Spectrogram: *When was each frequency strong?*

Large EEG waves do not necessarily mean that the brain is globally more active. They can also occur when many neurons become synchronized, causing their electrical fields to add together.

## 2. Data and experimental periods

The first NWB file contains one approximately 77-minute recording from mouse 688277. The EEG was sampled at approximately 2,500 Hz using a 30-channel skull-surface array. Three channels were marked invalid in the metadata.

| Condition | Labeled period in full recording | Selected window | Status |
|---|---:|---:|---|
| Awake | approximately 319–1,990 s | 600–660 s | Ready |
| Isoflurane | approximately 2,781–4,451 s | 3,000–3,060 s | Ready |
| Urethane | To be confirmed from second recording | — | Pending |
| Psilocybin | To be confirmed from second recording | — | Pending |
| Ketanserin | To be confirmed from second recording | — | Pending |

The selected windows are well inside their labeled periods rather than near the transition into anesthesia. Electrical pulses were delivered approximately every seven seconds during both windows. Those times are retained so that stimulation artifacts and evoked responses are not mistaken for spontaneous EEG activity.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import resample_poly, spectrogram, welch

# This works whether the notebook is launched from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = PROJECT_ROOT / 'data/raw/688277_eeg_extract.npz'
METADATA_FILE = PROJECT_ROOT / 'data/raw/688277_eeg_extract_metadata.json'

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f'{DATA_FILE} is unavailable. If it has a cloud icon in Finder, choose Download Now.'
    )

metadata = json.loads(METADATA_FILE.read_text())
with np.load(DATA_FILE) as saved:
    channel_ids = saved['channel_ids'].copy()
    channel_names = saved['channel_locations'].astype(str)
    conditions = {
        condition: {
            'time_s': saved[f'{condition}_time_s'].copy(),
            'eeg_uV': saved[f'{condition}_eeg_uV'].copy(),
            'stimulus_time_s': saved[f'{condition}_stimulus_time_s'].copy(),
        }
        for condition in metadata['windows']
    }

fs = float(metadata['sampling_rate_hz'])
print(f'Sampling rate: {fs:.2f} Hz')
print('Channels:', dict(zip(channel_ids.tolist(), channel_names.tolist())))
for condition, values in conditions.items():
    print(condition, values['eeg_uV'].shape)

## 3. Channel selection

Five valid electrodes were selected from different cortical locations to provide broader spatial coverage:

| Channel | Region | Approximate role |
|---:|---|---|
| 0 | VISl | Lateral visual cortex |
| 6 | SSp-bfd | Whisker-related somatosensory cortex |
| 12 | MOs | Secondary motor cortex |
| 21 | RSPagl | Retrosplenial cortex |
| 25 | VISp | Primary visual cortex |

A channel is most sensitive to activity near its electrode, but EEG fields spread through brain tissue and the skull. These labels provide spatial context rather than perfectly isolating one brain region.

## 4. Raw EEG traces

The raw traces below show 60 seconds from each condition. The channels are centered and vertically offset so they do not overlap. The same channel spacing is used across conditions, making their amplitudes more comparable. Red dashed lines mark electrical-stimulus times.

The data are properly resampled from approximately 2,500 Hz to 500 Hz for display only. All spectral calculations use the original sampling rate.

In [ ]:
colors = plt.cm.tab10(np.arange(len(channel_names)))
centered_data = {
    name: values['eeg_uV'] - np.median(values['eeg_uV'], axis=0)
    for name, values in conditions.items()
}
shared_amplitude = max(
    np.percentile(np.abs(values), 99) for values in centered_data.values()
)
channel_spacing = max(250.0, 2.2 * shared_amplitude)
downsample_factor = max(1, int(round(fs / 500)))

fig, axes = plt.subplots(len(conditions), 1, figsize=(14, 4 * len(conditions)), sharex=True)
axes = np.atleast_1d(axes)
for ax, (condition, values) in zip(axes, conditions.items()):
    displayed = resample_poly(centered_data[condition], 1, downsample_factor, axis=0)
    displayed_time = np.arange(len(displayed)) / (fs / downsample_factor)
    for index, channel_name in enumerate(channel_names):
        ax.plot(
            displayed_time,
            displayed[:, index] + index * channel_spacing,
            color=colors[index], lw=0.6,
        )
    for stimulus_time in values['stimulus_time_s']:
        ax.axvline(stimulus_time, color='crimson', alpha=0.35, ls='--', lw=0.8)
    ax.set_yticks(np.arange(len(channel_names)) * channel_spacing, channel_names)
    ax.set_ylabel('Channel')
    ax.set_title(f'{condition.capitalize()}: raw EEG')
    ax.grid(axis='x', alpha=0.2)
axes[-1].set_xlabel('Time within selected window (s)')
fig.suptitle('Raw EEG; red dashed lines indicate electrical stimuli')
plt.tight_layout()
plt.show()

### Initial raw-trace observation

The awake EEG appears relatively continuous and irregular. Under isoflurane, the recording appears to alternate between quieter intervals and large, synchronized events. This is more informative than saying that isoflurane simply makes the signal smaller: the organization of activity changes, and large bursts can coexist with suppressed intervals.

Several large events occur near electrical stimuli. The immediate sharp deflection may include electrical artifact, while later activity may contain a genuine network response. Some bursts may also be spontaneous.

## 5. Welch power spectral density

A power spectrum describes how the signal's variance is distributed across frequencies. Frequency is shown on the horizontal axis, and power is shown on the vertical axis.

Welch's method divides each 60-second recording into overlapping four-second sections, applies a taper to reduce edge effects, calculates a spectrum for each section, and combines the spectra. Here the median is used so that occasional large artifacts have less influence. Four-second sections provide approximately 0.25 Hz frequency resolution.

Power is displayed in decibels using `10 × log10(power)`, which compresses a wide numerical range. The original unit is microvolts squared per hertz.

In [ ]:
fig, axes = plt.subplots(1, len(conditions), figsize=(7 * len(conditions), 4), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, (condition, eeg_uv) in zip(axes, centered_data.items()):
    for index, channel_name in enumerate(channel_names):
        frequency, power = welch(
            eeg_uv[:, index],
            fs=fs,
            nperseg=int(4 * fs),
            noverlap=int(2 * fs),
            detrend='constant',
            average='median',
        )
        keep = (frequency >= 0.5) & (frequency <= 100)
        power_db = 10 * np.log10(power[keep] + np.finfo(float).tiny)
        ax.plot(frequency[keep], power_db, color=colors[index], label=channel_name)
    ax.axvline(60, color='gray', ls=':', lw=0.8, label='60 Hz')
    ax.set_title(condition.capitalize())
    ax.set_xlabel('Frequency (Hz)')
    ax.set_xlim(0.5, 100)
    ax.grid(alpha=0.2)
axes[0].set_ylabel('Power (dB µV²/Hz)')
axes[-1].legend(frameon=False, fontsize=8, ncol=2)
fig.suptitle('Welch power spectral density')
plt.tight_layout()
plt.show()

### Initial PSD observation

The awake channels had their strongest low-frequency peaks around 2.25–4.75 Hz, while the isoflurane channels peaked around 1–1.5 Hz. This is consistent with a shift toward slower population activity under anesthesia.

This is a descriptive comparison from one mouse. The PSD averages over the full minute and therefore cannot reveal when a rhythm occurred. It also includes stimulus-related activity, so it should not be interpreted as a completely spontaneous resting-state spectrum.

## 6. Spectrogram

A spectrogram preserves time information that is lost in the Welch PSD. Time is shown horizontally, frequency vertically, and color represents power. A persistent rhythm appears as a horizontal band, while a brief sharp event spreads across many frequencies and appears as a vertical stripe.

The `MOs` channel is used because the electrical stimulation was delivered to secondary motor cortex. Both conditions use the same color scale.

In [ ]:
representative_channel = 'MOs'
channel_index = int(np.flatnonzero(channel_names == representative_channel)[0])
spectra = {}
visible_power = []

for condition, eeg_uv in centered_data.items():
    frequency, time, power = spectrogram(
        eeg_uv[:, channel_index],
        fs=fs,
        window='hann',
        nperseg=int(2 * fs),
        noverlap=int(1.5 * fs),
        detrend='constant',
        scaling='density',
        mode='psd',
    )
    keep = (frequency >= 0.5) & (frequency <= 100)
    power_db = 10 * np.log10(power[keep] + np.finfo(float).tiny)
    spectra[condition] = (frequency[keep], time, power_db)
    visible_power.append(power_db.ravel())

shared_power = np.concatenate(visible_power)
color_min, color_max = np.percentile(shared_power, [5, 98])
fig, axes = plt.subplots(len(conditions), 1, figsize=(14, 4 * len(conditions)), sharex=True)
axes = np.atleast_1d(axes)
image = None
for ax, (condition, values) in zip(axes, conditions.items()):
    frequency, time, power_db = spectra[condition]
    image = ax.pcolormesh(
        time, frequency, power_db, shading='auto', cmap='magma',
        vmin=color_min, vmax=color_max,
    )
    for stimulus_time in values['stimulus_time_s']:
        ax.axvline(stimulus_time, color='cyan', alpha=0.75, ls='--', lw=0.9)
    ax.set_ylim(0.5, 100)
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{condition.capitalize()} — {representative_channel}')
axes[-1].set_xlabel('Time within selected window (s)')
fig.colorbar(image, ax=axes, label='Power (dB µV²/Hz)')
fig.suptitle('EEG spectrogram; cyan lines indicate electrical stimuli')
plt.show()

### Initial spectrogram observation

The awake recording shows comparatively continuous low-frequency power. The isoflurane recording contains darker intervals interrupted by broadband vertical events. This supports the description of alternating suppressed and burst-like activity rather than uniform dampening.

The two-second spectrogram windows blur timing. A colored event that appears slightly before or after a stimulus line does not prove a multi-second neural delay. Precise latency requires stimulus-aligned raw EEG.

## 7. Planned stimulus-aligned analysis

To determine whether responses are consistently delayed or simply variable, each usable stimulus will be aligned to time zero using a window from one second before to three seconds after the pulse.

The final figure will contain:

1. A trial-by-time heatmap, where each row is one stimulus.
2. The average response across trials with uncertainty shading.
3. Identical scales for awake and isoflurane.

This is a planned follow-up analysis for the next work session. It will distinguish a consistently delayed evoked response from variable or spontaneous burst activity.

## 8. Limitations

- This is one recording from one mouse, so it cannot establish a population-level drug effect.
- Electrical stimulation occurred throughout the selected windows and can influence both raw traces and spectra.
- Immediate stimulus artifacts must be distinguished from biological responses.
- EEG channels provide broad spatial sensitivity rather than perfectly isolated brain regions.
- Differences between separate mice may reflect animal, electrode-placement, or recording differences in addition to drugs.
- Welch PSD shows average frequency content but removes information about when activity occurred.
- Spectrogram timing is limited by the selected two-second analysis window.

## 9. Remaining work

1. Inspect the metadata of `692644-20230907-v001.nwb`.
2. Confirm the exact time ranges for urethane, psilocybin, and ketanserin rather than assuming them.
3. Extract matched 60-second windows and the same valid channel locations where possible.
4. Add raw EEG, Welch PSD, and spectrogram results for those conditions.
5. Create one final PSD overlay comparing all available conditions using identical parameters.
6. Run the stimulus-aligned analysis and separate immediate artifacts from later network activity.
7. Execute every notebook cell and verify all saved outputs before submission.

## 10. Preliminary conclusion

In this example session, awake EEG was relatively continuous, whereas isoflurane EEG appeared more organized into quieter intervals and large synchronized events. The average spectrum shifted toward slower frequencies under isoflurane. These observations suggest a change in the temporal organization and synchronization of population activity, not merely a uniform reduction in signal amplitude. Because electrical stimulation was present and only one mouse has been analyzed, the findings should remain descriptive until stimulus-aligned responses and additional sessions are included.